In [3]:
import sys, pandas, sklearn, shap
print(sys.executable)
print(pandas.__version__, sklearn.__version__, shap.__version__)

/Users/ayumumizushima/cardio-risk/.venv/bin/python3
3.0.6 1.9.1 0.53.0


In [4]:
import os
print(os.listdir("../data"))

[]


In [5]:
import os
print(os.listdir("../data"))

['extention of Z-Alizadeh sani dataset.xlsx']


In [6]:
import os
import pandas as pd

files = [f for f in os.listdir("../data") if f.lower().endswith((".xlsx", ".xls", ".csv"))]
print("Found:", files)

path = os.path.join("../data", files[0])
if path.lower().endswith(".csv"):
    df = pd.read_csv(path)
else:
    df = pd.read_excel(path)

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()

Found: ['extention of Z-Alizadeh sani dataset.xlsx']
Shape: (303, 59)
['Age', 'Weight', 'Length', 'Sex', 'BMI', 'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'BP', 'PR', 'Edema', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur', 'Typical Chest Pain', 'Dyspnea', 'Function Class', 'Atypical', 'Nonanginal', 'Exertional CP', 'LowTH Ang', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'BBB', 'FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE', 'Region RWMA', 'VHD', 'LAD', 'LCX', 'RCA', 'Cath']


,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,FH,...,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD,LAD,LCX,RCA,Cath
0,53,90,175,Male,29.387755,0,1,1,0,0,...,39,52,261,50,0,N,Stenotic,Normal,Stenotic,CAD
1,67,70,157,Fmale,28.398718,0,1,0,0,0,...,38,55,165,40,4,N,Stenotic,Stenotic,Normal,CAD
2,54,54,164,Male,20.077335,0,0,1,0,0,...,38,60,230,40,2,mild,Stenotic,Normal,Normal,CAD
3,66,67,158,Fmale,26.838648,0,1,0,0,0,...,18,72,742,55,0,Severe,Normal,Normal,Normal,Normal
4,50,87,153,Fmale,37.165193,0,1,0,0,0,...,55,39,274,50,0,Severe,Normal,Normal,Normal,Normal


In [7]:
print(df.dtypes)
print(df.isna().sum())

Age                        int64
Weight                     int64
Length                     int64
Sex                          str
BMI                      float64
DM                         int64
HTN                        int64
Current Smoker             int64
EX-Smoker                  int64
FH                         int64
Obesity                      str
CRF                          str
CVA                          str
Airway disease               str
Thyroid Disease              str
CHF                          str
DLP                          str
BP                         int64
PR                         int64
Edema                      int64
Weak Peripheral Pulse        str
Lung rales                   str
Systolic Murmur              str
Diastolic Murmur             str
Typical Chest Pain         int64
Dyspnea                      str
Function Class             int64
Atypical                     str
Nonanginal                   str
Exertional CP                str
LowTH Ang 

In [8]:
# 1) Missing values overall
print("Total missing:", int(df.isna().sum().sum()))
print(df.isna().sum()[df.isna().sum() > 0])
print()

# 2) Unique values in every non-numeric column
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in text_cols:
    print(c, "->", sorted(df[c].dropna().astype(str).unique().tolist()))
print()

# 3) How balanced are the four targets?
for t in ["Cath", "LAD", "LCX", "RCA"]:
    print(t, df[t].value_counts().to_dict())

Total missing: 0
Series([], dtype: int64)

Sex -> ['Fmale', 'Male']
Obesity -> ['N', 'Y']
CRF -> ['N', 'Y']
CVA -> ['N', 'Y']
Airway disease -> ['N', 'Y']
Thyroid Disease -> ['N', 'Y']
CHF -> ['N', 'Y']
DLP -> ['N', 'Y']
Weak Peripheral Pulse -> ['N', 'Y']
Lung rales -> ['N', 'Y']
Systolic Murmur -> ['N', 'Y']
Diastolic Murmur -> ['N', 'Y']
Dyspnea -> ['N', 'Y']
Atypical -> ['N', 'Y']
Nonanginal -> ['N', 'Y']
Exertional CP -> ['N']
LowTH Ang -> ['N', 'Y']
LVH -> ['N', 'Y']
Poor R Progression -> ['N', 'Y']
BBB -> ['LBBB', 'N', 'RBBB']
VHD -> ['Moderate', 'N', 'Severe', 'mild']
LAD -> ['Normal', 'Stenotic']
LCX -> ['Normal', 'Stenotic']
RCA -> ['Normal', 'Stenotic']
Cath -> ['CAD', 'Normal']

Cath {'CAD': 216, 'Normal': 87}
LAD {'Stenotic': 177, 'Normal': 126}
LCX {'Normal': 184, 'Stenotic': 119}
RCA {'Normal': 189, 'Stenotic': 114}


In [9]:
%%writefile ../preprocess.py
import pandas as pd

TARGET_COLS = ["Cath", "LAD", "LCX", "RCA"]   # never used as model inputs
VHD_ORDER = {"N": 0, "mild": 1, "Moderate": 2, "Severe": 3}

def clean_features(df):
    """Raw dataset rows -> all-numeric feature table. Target columns are always removed."""
    X = df.drop(columns=[c for c in TARGET_COLS if c in df.columns]).copy()
    X = X.drop(columns=["Exertional CP"], errors="ignore")   # constant column
    X["Sex"] = X["Sex"].map({"Male": 1, "Fmale": 0, "Female": 0})
    X["VHD"] = X["VHD"].map(VHD_ORDER)
    X["BBB_LBBB"] = (X["BBB"] == "LBBB").astype(int)
    X["BBB_RBBB"] = (X["BBB"] == "RBBB").astype(int)
    X = X.drop(columns=["BBB"])
    for c in X.columns:                      # remaining text columns are Y/N
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].map({"Y": 1, "N": 0})
    return X

def get_targets(df):
    """1 = CAD / Stenotic, 0 = Normal."""
    return {
        "CAD": (df["Cath"] == "CAD").astype(int),
        "LAD": (df["LAD"] == "Stenotic").astype(int),
        "LCX": (df["LCX"] == "Stenotic").astype(int),
        "RCA": (df["RCA"] == "Stenotic").astype(int),
    }

Writing ../preprocess.py


In [10]:
import sys
sys.path.append("..")
from preprocess import clean_features, get_targets

X = clean_features(df)
y = get_targets(df)

print(X.shape)
print("Missing after cleaning:", int(X.isna().sum().sum()))
print("Non-numeric columns left:", [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])])
print("Leaky columns present:", [c for c in ["Cath","LAD","LCX","RCA"] if c in X.columns])

(303, 55)
Missing after cleaning: 0
Non-numeric columns left: []
Leaky columns present: []


In [11]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
}
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

rows = []
for target, labels in y.items():
    for name, model in models.items():
        res = cross_validate(model, X, labels, cv=cv, scoring=scoring, n_jobs=-1)
        row = {"target": target, "model": name}
        row.update({m: round(res[f"test_{m}"].mean(), 3) for m in scoring})
        rows.append(row)

results = pd.DataFrame(rows)
results

,target,model,accuracy,precision,recall,f1,roc_auc
0,CAD,LogReg,0.835,0.916,0.847,0.879,0.909
1,CAD,RandomForest,0.870,0.908,0.912,0.909,0.927
2,LAD,LogReg,0.719,0.781,0.727,0.750,0.803
3,LAD,RandomForest,0.795,0.807,0.859,0.830,0.852
4,LCX,LogReg,0.603,0.500,0.568,0.528,0.640
5,LCX,RandomForest,0.683,0.607,0.563,0.582,0.725
6,RCA,LogReg,0.647,0.531,0.612,0.563,0.696
7,RCA,RandomForest,0.671,0.578,0.497,0.527,0.710


In [12]:
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

models2 = {
    "ExtraTrees": ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=42),
    "HistGB": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}

rows2 = []
for target, labels in y.items():
    for name, model in models2.items():
        res = cross_validate(model, X, labels, cv=cv, scoring=scoring, n_jobs=-1)
        row = {"target": target, "model": name}
        row.update({m: round(res[f"test_{m}"].mean(), 3) for m in scoring})
        rows2.append(row)

results = pd.concat([results, pd.DataFrame(rows2)], ignore_index=True)
results.sort_values(["target", "roc_auc"], ascending=[True, False])

,target,model,accuracy,precision,recall,f1,roc_auc
1,CAD,RandomForest,0.870,0.908,0.912,0.909,0.927
8,CAD,ExtraTrees,0.869,0.896,0.924,0.910,0.927
0,CAD,LogReg,0.835,0.916,0.847,0.879,0.909
9,CAD,HistGB,0.864,0.893,0.923,0.907,0.905
3,LAD,RandomForest,0.795,0.807,0.859,0.830,0.852
10,LAD,ExtraTrees,0.795,0.793,0.885,0.835,0.839
11,LAD,HistGB,0.753,0.782,0.810,0.793,0.808
2,LAD,LogReg,0.719,0.781,0.727,0.750,0.803
13,LCX,HistGB,0.690,0.632,0.516,0.565,0.736
5,LCX,RandomForest,0.683,0.607,0.563,0.582,0.725


In [13]:
import joblib
from sklearn.base import clone

all_models = {**models, **models2}
best = results.loc[results.groupby("target")["roc_auc"].idxmax()]
print(best[["target", "model", "accuracy", "f1", "roc_auc"]])

for _, r in best.iterrows():
    m = clone(all_models[r["model"]])
    m.fit(X, y[r["target"]])
    joblib.dump(m, f"../models/{r['target']}.joblib")

joblib.dump(list(X.columns), "../models/feature_columns.joblib")
results.to_csv("../models/cv_results.csv", index=False)
print("Saved:", sorted(os.listdir("../models")))

   target         model  accuracy     f1  roc_auc
1     CAD  RandomForest     0.870  0.909    0.927
3     LAD  RandomForest     0.795  0.830    0.852
13    LCX        HistGB     0.690  0.565    0.736
14    RCA    ExtraTrees     0.668  0.456    0.732
Saved: ['CAD.joblib', 'LAD.joblib', 'LCX.joblib', 'RCA.joblib', 'cv_results.csv', 'feature_columns.joblib']


In [14]:
rf_lcx = clone(models["RandomForest"])
rf_lcx.fit(X, y["LCX"])
joblib.dump(rf_lcx, "../models/LCX.joblib")

# Sanity check: predicted probabilities for the first patient in the dataset
import numpy as np
row = X.iloc[[0]]
for t in ["CAD", "LAD", "LCX", "RCA"]:
    m = joblib.load(f"../models/{t}.joblib")
    print(t, "probability:", round(float(m.predict_proba(row)[0, 1]), 3), "| actual:", int(y[t].iloc[0]))

CAD probability: 0.79 | actual: 1
LAD probability: 0.697 | actual: 1
LCX probability: 0.113 | actual: 0
RCA probability: 1.0 | actual: 1
